In [1]:
import warnings
from pathlib import Path

import numpy as np
import xarray as xr
import prism

from imagematerials import vehicles as vhc
from imagematerials import buildings as bld
from imagematerials.concepts import create_vehicle_graph, create_building_graph

from imagematerials.factory import ModelFactory, Sector
from imagematerials.maintenance import Maintenance
from imagematerials.model import (
    EndOfLife,
    GenericMainModel,
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
)
from imagematerials.util import (
    export_to_netcdf,
    import_from_netcdf,
    read_circular_economy_config,
    read_climate_policy_config,
    rebroadcast_prep_data,
)

from imagematerials.eol import preprocess_eol

base_dir = "../data/raw"
climate_policy_scenario_dir = Path(base_dir) / 'SSP2'
circular_economy_scenario_dirs = {"base": Path(base_dir) / 'circular_economy_scenarios' / 'base'}
prep_fp = Path("prep_vema.nc")
# note: configuration currently only works when no prep file is saved



# EoL
prep_eol = preprocess_eol("../../data/raw")

# Vehicles
def get_vema_prep():
    base_dir = Path("..", "data", "raw")
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        climate_policy_scenario_dir = base_dir / 'SSP2'
        circular_economy_scenario_dirs = {"slow": base_dir / 'circular_economy_scenarios' / 'slow'}

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            climate_policy_config = read_climate_policy_config(climate_policy_scenario_dir)
            circular_economy_config = read_circular_economy_config(circular_economy_scenario_dirs)
            prep_data = vhc.preprocess(base_dir, climate_policy_config, circular_economy_config)

        export_to_netcdf(prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)

    target_materials = [
    "Aluminium", "Brick", "Cement", "Concrete", 
    "Copper", "Glass", "Steel", "Wood"
]
    
    prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['battery_materials'] = prep_data['battery_materials'].reindex(material=target_materials)
    prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['material_fractions'] = prep_data['material_fractions'].reindex(material=target_materials)
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
    prep_data['maintenance_material_fractions'] = prep_data['maintenance_material_fractions'].reindex(material=target_materials)
    
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph
    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    new_prep_data["shares"] = None
    sec_vhc = Sector("vhc", new_prep_data)
    return sec_vhc
    

# Buildings

def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = bld.preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    new_prep_data["shares"] = None
    sec_bld = Sector("bld", new_prep_data)
    return sec_bld


sec_vhc = get_vema_prep() 
sec_bld = get_buildings_prep()
eol_sector = Sector(name="eol", data = prep_eol)

sectors = [sec_vhc, sec_bld,eol_sector]

# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")


factory = ModelFactory(sectors, complete_timeline)
factory.add(GenericStocks, ["vhc","bld"])
factory.add(GenericMaterials, "vhc")
factory.add(MaterialIntensities,"bld")

factory.add(EndOfLife, "eol", input_sources={
    "outflow_by_cohort_materials": ["vhc", "bld"],
    "collection": "eol",
    "reuse": "eol",
    "recycling": "eol"}
)

model = factory.finish()

In [2]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [16]:
model.bld['outflow_by_cohort_materials']

TimeVariable(
  timeline=Timeline(start=1960, end=2060, stepsize=1),
  unit=count,
  dims=
	Dimension(label="Region", coords=[np.str_('1'), np.str_('2'), np.str_('3'),
	  np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'),
	  np.str_('9'), np.str_('10'), np.str_('11'), np.str_('12'),
	  np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'),
	  np.str_('17'), np.str_('18'), np.str_('19'), np.str_('20'),
	  np.str_('21'), np.str_('22'), np.str_('23'), np.str_('24'),
	  np.str_('25'), np.str_('26')])
        Dimension(label="Type",
	  coords=[np.str_('Appartment - Rural'), np.str_('Appartment - Urban'),
	  np.str_('Detached - Rural'), np.str_('Detached - Urban'), np.str_('High-
	  rise - Rural'), np.str_('High-rise - Urban'), np.str_('Semi-detached -
	  Rural'), np.str_('Semi-detached - Urban'), np.str_('Office'),
	  np.str_('Retail+'), np.str_('Hotels+'), np.str_('Govt+')])
	  Dimension(label="material", coords=[np.str_('Aluminium'),
	  np.str_('Brick'), np.str_('C

In [ ]:
# tests
# prep_data = import_from_netcdf(prep_fp)
# # prep_data['material_fractions'] = prep_data['material_fractions'].assign_coords(
# #     material=['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #               'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'])
# # prep_data['lifetimes'].material.values
# prep_data.keys()



# # prep_data['material_fractions'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #       'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )
# #.assign_coords('material' = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb',
# #       'Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'])
# prep_data['maintenance_material_fractions']
# prep_data['battery_materials'] = prep_data['battery_materials'].assign_coords(material = ['Aluminium', 'Co', 'Copper', 'Glass', 'Li', 'Mn', 'Nd', 'Ni', 'Pb','Plastics', 'Rubber', 'Steel', 'Ti', 'Wood'] )

# prep_data['battery_materials'].sel(material = "Copper") # deu certo!!!


In [ ]:

sec_bld.all_data

{'lifetimes': {'weibull': <xarray.DataArray 'weibull' (Time: 340, Region: 26, Type: 12, ScipyParam: 2)> Size: 2MB
  array([[[[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
           [  1.96820464,  57.52862846],
           [  1.96820464,  57.52862846],
           [  1.96820464,  57.52862846]],
  
          [[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
           [  4.16343417,  85.18683893],
           [  4.16343417,  85.18683893],
           [  4.16343417,  85.18683893]],
  
          [[  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           [  1.44      ,  49.57      ],
           ...,
  ...
           ...,
           [  1.96820464, 122.2       ],
           [  1.96820464, 122.2       ],
           [  1.96820464, 122.2       ]],
  
          [[  1.44      ,  94.18      ],
           [  1.44      ,

In [ ]:
model.bld['outflow_by_cohort_materials']

TimeVariable(
  timeline=Timeline(start=1960, end=2060, stepsize=1),
  unit=count,
  dims=
	Dimension(label="Region", coords=[np.str_('1'), np.str_('2'), np.str_('3'),
	  np.str_('4'), np.str_('5'), np.str_('6'), np.str_('7'), np.str_('8'),
	  np.str_('9'), np.str_('10'), np.str_('11'), np.str_('12'),
	  np.str_('13'), np.str_('14'), np.str_('15'), np.str_('16'),
	  np.str_('17'), np.str_('18'), np.str_('19'), np.str_('20'),
	  np.str_('21'), np.str_('22'), np.str_('23'), np.str_('24'),
	  np.str_('25'), np.str_('26')])
        Dimension(label="Type",
	  coords=[np.str_('Appartment - Rural'), np.str_('Appartment - Urban'),
	  np.str_('Detached - Rural'), np.str_('Detached - Urban'), np.str_('High-
	  rise - Rural'), np.str_('High-rise - Urban'), np.str_('Semi-detached -
	  Rural'), np.str_('Semi-detached - Urban'), np.str_('Office'),
	  np.str_('Retail+'), np.str_('Hotels+'), np.str_('Govt+')])
	  Dimension(label="material", coords=[np.str_('Aluminium'),
	  np.str_('Brick'), np.str_('C

In [ ]:
model.eol['sum_outflow'][2020]

Magnitude,[[[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] ... [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]] [[0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] ... [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0] [0.0 0.0 0.0 ... 0.0 0.0 0.0]]]
Units,count


In [ ]:
print('outflow_by_cohort_materials'[t])


NameError: name 't' is not defined

In [ ]:
model.vhc['outflow_by_cohort_materials'][2020]

Magnitude,[[[-74494.77666666429 0.0 0.0 ... 0.0 -55061.35666666494 0.0] [1299418073.6566057 0.0 0.0 ... 0.0 171981803.86631548 0.0] [635068.0461390927 0.0 0.0 ... 0.0 11087278.550432032 0.0] ... [-1901725.7015534071 0.0 0.0 ... -224439.70344992785 -5544863.030767413 0.0] [-1901725.7015534071 0.0 0.0 ... -224439.70344992785 -5544863.030767413 0.0] [-1901725.7015534071 0.0 0.0 ... -224439.70344992785 -5544863.030767413 0.0]] [[-304644.966666691 0.0 0.0 ... 0.0 -225172.36666668532 0.0] [48292766380.92855 0.0 0.0 ... 0.0 6391689668.064073 0.0] [15901631.933240138 0.0 0.0 ... 0.0 277617215.54427564 0.0] ... [-21219713.05942662 0.0 0.0 ... -2504328.622398083 -61870333.019424066 0.0] [-21219713.05942662 0.0 0.0 ... -2504328.622398083 -61870333.019424066 0.0] [-21219713.05942662 0.0 0.0 ... -2504328.622398083 -61870333.019424066 0.0]] [[1.0408181697130204e-07 0.0 0.0 ... 0.0 7.693003863096237e-08 0.0] [682484281.5663631 0.0 0.0 ... 0.0 90328801.97201866 0.0] [1243172.464769774 0.0 0.0 ... 0.0 21703815.027265277 0.0] ... [-16371481.646501265 0.0 0.0 ... -1932145.3576482004 -47734341.11261222 0.0] [-16371481.646501265 0.0 0.0 ... -1932145.3576482004 -47734341.11261222 0.0] [-16371481.646501265 0.0 0.0 ... -1932145.3576482004 -47734341.11261222 0.0]] ... [[-70179.44 0.0 0.0 ... 0.0 -51871.76 0.0] [2131655015.1542535 0.0 0.0 ... 0.0 282130810.82923937 0.0] [1562978.3729762358 0.0 0.0 ... 0.0 27287117.805472363 0.0] ... [-5226787.472471716 0.0 0.0 ... -616860.0599755866 -15239748.088861153 0.0] [-5226787.472471716 0.0 0.0 ... -616860.0599755866 -15239748.088861153 0.0] [-5226787.472471716 0.0 0.0 ... -616860.0599755866 -15239748.088861153 0.0]] [[-23275455.66666673 0.0 0.0 ... 0.0 -17203597.666666716 0.0] [593910344.0521762 0.0 0.0 ... 0.0 78605780.83043507 0.0] [36433.54759662516 0.0 0.0 ... 0.0 636071.8244919125 0.0] ... [-50165653.81835508 0.0 0.0 ... -5920498.659278996 -146268033.84129453 0.0] [-50165653.81835508 0.0 0.0 ... -5920498.659278996 -146268033.84129453 0.0] [-50165653.81835508 0.0 0.0 ... -5920498.659278996 -146268033.84129453 0.0]] [[2054222.8414283297 0.0 0.0 ... 0.0 1518338.621925287 0.0] [100080808.60415082 0.0 0.0 ... 0.0 13245989.37407878 0.0] [-424432.1229201899 0.0 0.0 ... 0.0 -7409910.168172261 0.0] ... [-30194799.416492835 0.0 0.0 ... -3563559.045992831 -88038998.93091218 0.0] [-30194799.416492835 0.0 0.0 ... -3563559.045992831 -88038998.93091218 0.0] [-30194799.416492835 0.0 0.0 ... -3563559.045992831 -88038998.93091218 0.0]]]
Units,count


In [ ]:
model.bld['outflow_by_cohort_materials'][2020]

<xarray.DataArray (Region: 26, Type: 12, material: 8)> Size: 20kB
<Quantity([[[ 6.37262001e-01  5.04488135e+01  0.00000000e+00 ...  2.08588335e+00
    3.19813549e+01  1.22098085e+01]
  [-3.11659394e+00 -2.46724999e+02  0.00000000e+00 ... -1.02012224e+01
   -1.56408034e+02 -5.97132973e+01]
  [ 3.46384997e+01  2.01356512e+03  0.00000000e+00 ...  1.44596852e+01
    1.75188798e+02  2.74680065e+02]
  ...
  [ 4.08157361e-01  3.31998908e+01  7.29859572e+01 ...  3.33946932e-01
    7.42104293e+00  2.38401004e+00]
  [ 1.56608714e-01  1.23818765e+01  6.67870912e+00 ...  1.72922122e-01
    3.41602758e+00  2.51226479e-01]
  [ 4.26927186e-01  8.28594513e+01  4.80115198e+01 ...  7.64911208e-01
    1.26655065e+01  3.25531979e+00]]

 [[ 1.60800890e+00  7.19994190e+01  0.00000000e+00 ...  2.97692610e+00
    2.31121979e+01  1.74255659e+01]
  [-2.56092382e+01 -1.14666670e+03  0.00000000e+00 ... -4.74106887e+01
   -3.68086135e+02 -2.77520520e+02]
  [ 7.51860507e+01  1.03923830e+04  0.00000000e+00 ...  8.35400564e+01
    5.55541375e+02  1.41349775e+03]
...
  [ 2.63609362e-03  2.14422251e-01  4.71381468e-01 ...  2.15680387e-03
    4.79289749e-02  1.53971832e-02]
  [ 6.34372172e-03  5.01550498e-01  2.70533299e-01 ...  7.00452606e-03
    1.38372430e-01  1.01763869e-02]
  [ 5.72732460e-02  1.11157825e+01  6.44085379e+00 ...  1.02614566e-01
    1.69910630e+00  4.36708501e-01]]

 [[-2.82756729e-01 -2.23844219e+01  0.00000000e+00 ... -9.25518158e-01
   -1.41903068e+01 -5.41756062e+00]
  [-4.74003190e+00 -3.75244381e+02  0.00000000e+00 ... -1.55150529e+01
   -2.37881189e+02 -9.08180340e+01]
  [-8.52253844e+00 -8.93430153e+02  0.00000000e+00 ... -6.41584354e+00
   -7.81152891e+01 -1.27048066e+02]
  ...
  [ 2.28483497e-03  1.85850554e-01  4.08570036e-01 ...  1.86941043e-03
    4.15424540e-02  1.33455134e-02]
  [ 3.98571329e-03  3.15120457e-01  1.69974065e-01 ...  4.40089176e-03
    8.69383712e-02  6.39374841e-03]
  [ 2.77225654e-02  5.38048790e+00  3.11763350e+00 ...  4.96695963e-02
    8.22436106e-01  2.11384561e-01]]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * Type      (Type) <U21 1kB 'Appartment - Rural' ... 'Govt+'
  * material  (material) <U9 288B 'Aluminium' 'Brick' ... 'Steel' 'Wood'